In [5]:
#importing requried libraies
from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd
from bs4 import BeautifulSoup
import requests

In [6]:
#setting upwebdriver for firefox 
service=Service(r'C:\users\asuna\Downloads\geckodriver.exe') #different for different devices
options=Options()
options.add_argument('--headless')
options.add_argument('--window-size=1920,1080')
options.add_argument("--disable-blink-features=AutomationControlled") #for preventing bot detection
driver= webdriver.Firefox(service=service,options=options)

In [8]:
url='https://qatarcid.com/listings/investment/'   #url need to be changed based on the category 
driver.get(url)
time.sleep(5)  # buffer time
data = []

In [11]:
#function for detecting all listings in main page
def containers():  
    container = driver.find_element(By.CLASS_NAME, 'pfitemlists-content-elements')
    listings=container.find_elements(By.CLASS_NAME,'pflisting-itemband')
    return listings

In [15]:
#listings=containers()

In [27]:
# function to scrape multi pages
def scrape():
    WebDriverWait(driver, 10).until(lambda d: d.execute_script("return document.readyState") == "complete")
    time.sleep(4)
    listings=containers()
    for index, listing in enumerate(listings):
        listing_url = listing.find_element(By.TAG_NAME, 'a').get_attribute('href')
    
        # Fetch page content
        headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"}
        response = requests.get(listing_url, headers=headers)
    
    
        soup = BeautifulSoup(response.text, 'html')
    
        # Extracting company name and address from title
        title_bar = soup.find('div', class_='pf-item-title-bar')
        
        if title_bar:
            company_name = title_bar.find('h1').text.strip() if title_bar.find('h1') else None
            address_element = title_bar.find('div', class_='pf-item-subtitle')
            address = address_element.text.strip() if address_element else None
        else:
            company_name = None
            address = None
    
        # Extract additional details
        details_section = soup.find('div', class_='pf-itempage-details')
        details = details_section.find_all('div', class_='pfdetailitem-subelement') if details_section else []
    
        detail_dict = {
            'Company Name': company_name,
            'Address': address,
            'Listing Type': None,
            'Location': None,
            'QCCI Membership Number': None,
            'CR Number': None,
            'Company Type': None,
            'Address Detail': None,
            'PO Box': None,
            'Phone': None,
            'Email': None,
            'Website': None,
            'Contact Person Mobile': None,
            'Contact Person': None,
            'Owner Name': None
        }
    
        for detail in details:
            title_element = detail.find('span', class_='pf-ftitle')
            value_element = detail.find('span', class_='pfdetail-ftext')
    
            if title_element and value_element:
                title = title_element.text.strip().replace(' :', '')
                value = value_element.text.strip()
                if title in detail_dict:
                    detail_dict[title] = value
    
        data.append(detail_dict)

In [6]:
page=1
while True:
    
    scrape()
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    next_buttons = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, f"//a[contains(@class, 'page-numbers') and text()='{page + 1}']")))
    # Wait for new content to load
    time.sleep(2)
    if next_buttons:
        page += 1
        next_buttons.click()
        print(f"Navigated to Page {page}")
        # Wait until document.readyState returns "complete"
        WebDriverWait(driver, 10).until(lambda d: d.execute_script("return document.readyState") == "complete")
        
    else:
        print(f"No more pages after {page_number}. Exiting.")
        break

Navigated to Page 539
Navigated to Page 540
Navigated to Page 541
Navigated to Page 542
Navigated to Page 543
Navigated to Page 544
Navigated to Page 545
Navigated to Page 546
Navigated to Page 547
Navigated to Page 548
Navigated to Page 549
Navigated to Page 550
Navigated to Page 551
Navigated to Page 552
Navigated to Page 553
Navigated to Page 554
Navigated to Page 555
Navigated to Page 556
Navigated to Page 557
Navigated to Page 558
Navigated to Page 559
Navigated to Page 560
Navigated to Page 561
Navigated to Page 562
Navigated to Page 563
Navigated to Page 564
Navigated to Page 565
Navigated to Page 566
Navigated to Page 567
Navigated to Page 568
Navigated to Page 569
Navigated to Page 570
Navigated to Page 571
Navigated to Page 572
Navigated to Page 573
Navigated to Page 574
Navigated to Page 575
Navigated to Page 576
Navigated to Page 577
Navigated to Page 578
Navigated to Page 579
Navigated to Page 580
Navigated to Page 581
Navigated to Page 582
Navigated to Page 583
Navigated 

TimeoutException: Message: 
Stacktrace:
RemoteError@chrome://remote/content/shared/RemoteError.sys.mjs:8:8
WebDriverError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:197:5
NoSuchElementError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:527:5
dom.find/</<@chrome://remote/content/shared/DOM.sys.mjs:136:16


In [13]:
df=pd.DataFrame(data)

In [15]:
df.count()

Company Name              4611
Address                     30
Listing Type              4611
Location                    35
QCCI Membership Number    4609
CR Number                 4524
Company Type                22
Address Detail               0
PO Box                    4605
Phone                     4604
Email                      536
Website                    148
Contact Person Mobile       16
Contact Person            2065
Owner Name                  23
dtype: int64

In [17]:
df.to_csv("investment.csv", index=False)

In [19]:
df['Company Name'].nunique()

4609

In [21]:
driver.quit()